In [ ]:
import numpy as np
import pickle

with open('/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/Dill/collection_fc_allmice.pkl', 'rb') as file:
    collection_fc = pickle.load(file)

with open('/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/Dill/collection_cxt_allmice.pkl', 'rb') as file:
    collection_cxta = pickle.load(file)


In [ ]:
collection_cxtb.animals

In [ ]:
table = collection_fc.animals['astroM3'].registration_tables['FC-A']
table

In [ ]:
# rows where both FC and CxtA exist
valid_rows = np.where((table[:, 0] != -1) & (table[:, 1] != -1))[0]
valid_rows

In [ ]:
# collect the actual row indices for each recording
fc_idx     = table[valid_rows, 0].astype(int)
fc_idx

In [ ]:
recall_idx = table[valid_rows, 1].astype(int)
recall_idx

In [ ]:
fc_traces = collection_fc.animals['astroM3'].accepted_traces.to_numpy().T
fc_traces

In [ ]:
recall_traces = collection_cxta.animals['astroM3'].accepted_traces.to_numpy().T #this is cell x time
recall_traces

In [ ]:
fc_active     = fc_traces[fc_idx, :]
fc_active

In [ ]:
    # rows by position
recall_active = recall_traces[recall_idx, :]

In [ ]:
recall_active

In [ ]:
from scipy import stats
fc_dfz = stats.zscore(fc_active, axis=1)
recall_dfz = stats.zscore(recall_active, axis=1)

In [ ]:
import matplotlib.pyplot as plt
data = fc_dfz
plt.figure(figsize=(10, 6))
for i in range(data.shape[0]):
    plt.plot(data[i, :],label=f'Cell {i}')  # offset each cell vertically
plt.xlabel('Time (frames)')
plt.ylabel('ΔF/F (offset per cell)')
plt.title('Calcium traces across cells')
plt.tight_layout()
plt.show()

In [ ]:
import onep
time = collection_fc.animals['astroF7'].Timestamps.to_numpy().squeeze()

across_eta_, time_fc = onep.eta_individual_cells(
    data=fc_dfz,
    timestamps=time,
    events=[[120, 180, 240, 300]],  # detected onsets
    window=15
)

across_eta_R, time_R = onep.eta_individual_cells(
    data=recall_dfz,
    timestamps=time,
    events=[[]],
    window=15
)
#astroM3 - 71.9448772796669, 224.9276649396253, 266.7955865787648

#astroM4-121.5068386007773, 171.4685321043864, 243.9129876846197,
       #285.4811166796225

#astroM5 - 34.67342136063298, 89.83114064325375, 173.8667238256524

#astroM6 - 84.73501601221373,  148.286278021374, 174.9658172610687,
       #266.6954690290077

#astroM7 - 43.16693610899183,  118.509227373297, 229.5241950054496,
       #266.8955702479564

#astroM8 - 91.42997888754353,133.3978380490389,248.8094507431512,  275.489018352959

#astroM9 - 67.54823800666061, 111.5145467683924, 143.0903503336361

In [ ]:
time_fc

In [ ]:
across_eta_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- sort by Day 1 argmax ---
order = np.argmax(across_eta_, axis=1).argsort()
time= time_fc
fc_sorted = across_eta_[order]
recall_sorted = across_eta_R[order]

# --- figure aesthetics matching your original ---
sns.set(style="ticks", font="Arial")
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(7, 5), dpi=600)

vmin, vmax = -4, 4
first_last = [0, across_eta_.shape[0]]

# --- FC heatmap (Day 1) ---
sns.heatmap(fc_sorted, ax=ax1, cmap='mako', cbar=False,
            xticklabels=True, rasterized=True, yticklabels=True,
            vmin=vmin, vmax=vmax)
ax1.tick_params(axis='both', which='major', labelsize=8)
ax1.set_xticks(np.linspace(0, len(time), 5))
ax1.set_xticklabels(np.linspace(time.min(), time.max(), 5).astype(int), rotation=0)
ax1.set_yticks(first_last)
ax1.set_yticklabels([first_last[0], first_last[1]], rotation=0)
ax1.set_title('FC (sorted by argmax)', fontsize=10, pad=10)
ax1.set_ylabel('Cell #', fontsize=10)
ax1.set_xlabel('Time (frames)', fontsize=10)
sns.despine(ax=ax1, left=True, bottom=True)

# --- RECALL heatmap (Day 2) ---
sns.heatmap(recall_sorted, ax=ax2, cmap='mako', cbar=True,
            cbar_kws={'label': 'Z-score'}, xticklabels=True,
            rasterized=True, yticklabels=True, vmin=vmin, vmax=vmax)
ax2.tick_params(axis='both', which='major', labelsize=8)
ax2.set_xticks(np.linspace(0, len(time), 5))
ax2.set_xticklabels(np.linspace(time.min(), time.max(), 5).astype(int), rotation=0)
ax2.set_yticks(first_last)
ax2.set_yticklabels([first_last[0], first_last[1]], rotation=0)
ax2.set_title('RECALL (same order as FC)', fontsize=10, pad=10)
ax2.set_xlabel('Time (frames)', fontsize=10)
sns.despine(ax=ax2, left=True, bottom=True)

plt.tight_layout(pad=3)
plt.show()